In [1]:
def render_c(filename):
    from IPython.display import Markdown
    with open(filename) as f:
        contents = f.read()
    return Markdown("```c\n" + contents + "```\n")

# 20261015 Intro to MPI

## Processes and Threads

Threads and processes are very similar
* Both created via [clone system call](https://linux.die.net/man/2/clone) on Linux
    - Side note: Linux creates processes/threads in a very non-intuitive/interesting way. See [this video](https://youtu.be/SwIPOf2YAgI?si=NrBurVPOCZDhbFJp)
* Scheduled in the same way by the operating system
* Separate stacks (automatic variables)
* Access to same memory before `fork()` or `clone()`

...with some important distinctions

Flags passed to `clone` for threads:
* `CLONE_VM`
  * Threads share the same virtual-to-physical address mapping
  * Threads can access the same data at the same addresses
  * "Private data" is private only because other threads don't know its address
      - Possible for one thread to accidentally write over "private" data of another thread
      - Obviously a bug and probably undefined behavior as well

* `CLONE_FILES`
  * Threads share file descriptors
  * If one thread opens a file, all threads can access the file
* `CLONE_THREAD`, `CLONE_SIGHAND`
  * Process ID and signal handlers shared
  * Attempting to stop (e.g. `Ctrl + C` in terminal) one of the threads will stop all of the threads

### Myths told by Big-Thread
* Processes can't share memory
  * Memory is private by default
  * Can still share using OS utilities (`mmap()`, `shm_open()`) or other parallel framework utilities (`MPI_Win_allocate_shared()`)

* Processes are "heavy"
  * same data structures and kernel scheduling; no difference in context switching
  * data from parent is inherited copy-on-write at very low overhead
  * startup costs ~100 microseconds to duplicate page tables
  * caches are physically tagged; processes can share L1 cache

## MPI: Message Passing Interface
- API (soon to be ABI) for doing distributed memory computing
    - ABI allows for swaping MPI libraries without having to recompile
    - Particularly important for software distribution
   

- Implementations in the form of libraries
	- [MPICH](https://www.mpich.org/) and [Open MPI](https://www.open-mpi.org/) being the two primary ones
	- MPICH was the first one ever created (created before the MPI standard was finalized), OpenMPI is very common
    - [MVAPICH](https://mvapich.cse.ohio-state.edu/) also exists, but isn't as widely used for whatever reason
	- Many vendor-specific MPI libraries out there too
   

- Most implementations scalable to 100k+ compute nodes
  - Shared memory systems can be scaled up to [~4000 cores](https://www.uvhpc.com/sgi-uv-3000), but latency and price ($) increase
 

- Bindings in more-or-less every single language
	- C, Fortran, and Python (`mpi4py`) are the most mature and fully featured
	- Most other languages have bindings as well (Julia, Rust, etc.)
	- C++ has [MPL](https://github.com/rabauke/mpl), which is a wrapper around MPI for a more "modern C++" interface (cue jokes about what constitutes as "modern C++")
   

- Bindings are independent of which specific MPI library is used
    - At least the interface is
    - Example: `mpi4py` is not tied to OpenMPI or MPICH; you choose the specific library independent of that
    - Without stable ABI, technically have to recompile
        - Don't need to change Python code though

- In spite of the name, has come to faciliate more parallel programming paradigms than just message passing with newer versions
	- MPI-1 (1994) point-to-point messaging, collectives
	- MPI-2 (1997) parallel I/O, dynamic processes, one-sided (remote direct memory access, RDMA)
	- MPI-3 (2012) nonblocking collectives, neighborhood collectives, improved one-sided, shared memory
	- MPI-4 (2021) no major feature upgrades, minor API breaking changes (e.g. `int` -> `long`)
    - MPI-5 (2025) (no major feature upgrades, standardized ABI)
	- GPU-aware communication (be able to transfer data between GPUs without explicitly needing a copy to main memory)

- Since MPI is an API, the implementation of many of the features is dependent on the library and options associated with them
    - For example, [OpenMPI](https://docs.open-mpi.org/en/main/tuning-apps/collectives/tuned.html#fixed-forced-and-dynamic-decisions) allows you to specify the algorithm used with certain collective communication operations
    - Similar features in all MPI library implementations
    - Default choices are probably good enough
        - Use the MPI library provided by the HPC system
        - Hopefully they've done some benchmarking and have selected some good defaults

## Anatomy of MPI program

- MPI run as SPMD (single program, multiple data)
    - We have a single program that is literally cloned and run across multiple instances
    - Every process/rank runs the exact same program
- Perspective change from OpenMP task/fork-join to MPI
- OpenMP: Often, we talk about the programs from the perspective of the master thread
    - "Generate tasks and pass them to a team of threads" -- Spoken from perspective of the master thread
   

- MPI: Often spoken in first person as a "member" rank
    - "my rank", or "current rank" is from the perspective of a single rank
    - Effectively, "live" the life of each rank simultaneously
    - MPI programs often do have some "root rank" exclusive functionality, but generic rank-to-rank communication is the bread and butter

```c
int main() {
    #pragma omp parallel
    {
        int id = omp_get_thread_num();
        int num_threads = omp_get_num_threads();
        // your code
    }
}
```
Similar MPI code:

In [2]:
render_c('mpi-demo.c')

```c
#include <mpi.h>
#include <stdio.h>

int main(int argc, char **argv) {
  MPI_Init(&argc, &argv);   // Must call before any other MPI functions
  int size, rank, sum;
  MPI_Comm_rank(MPI_COMM_WORLD, &rank);
  MPI_Comm_size(MPI_COMM_WORLD, &size);
  MPI_Allreduce(&rank, &sum, 1, MPI_INT, MPI_SUM, MPI_COMM_WORLD);
  printf("I am rank %d of %d: sum=%d\n", rank, size, sum);
  return MPI_Finalize();
}
```


- Always started with `MPI_Init()` and ended with `MPI_Finalize()`
    - Side note: while `MPI_Init()` takes the `argc` and `argv` parameters, it doesn't actually read any of the input flags; it just modifies the flags to that they're in the appropriate locations (ignores the position of flags handed to `mpirun`/`mpiexec`
   

- Ranks collected into communication groups (`MPI_Comm`)
    - `MPI_COMM_WORLD` is the default comm, with every single rank
    - `MPI_COMM_SELF` is the comm defined exclusively for the "current" rank
- Each rank (process) is assigned a rank number

- Program started by a `mpirun`/`mpiexec` command, which
	- Starts up processes across the machine(s) you specify
	- Copies the program to each of those machines
	- Runs said program
	- Provides info for `MPI_Init()` to setup the parallel communication data structures
- Each processor is identified by a rank ID (similar to `omp_get_thread_num()`)
- Processors can be divided into sub-communicators
	- `MPI_COMM_WORLD` has all the processors in a single `mpirun` launch
	- `MPI_COMM_SELF` is the communicator with only the currently running process in it

* We use the compiler wrapper `mpicc`, but it just passes some flags to the real compiler.

In [3]:
! mpicc -show

gcc -I/usr/include -Wl,-rpath -Wl,/usr/lib -Wl,--enable-new-dtags -lmpi


In [4]:
! make CC=mpicc CFLAGS=-Wall mpi-demo

mpicc -Wall    mpi-demo.c   -o mpi-demo


* We use `mpiexec` to run locally.  Clusters/supercomputers often have different job launching programs. (e.g. `srun` on SLURM based machines)

In [7]:
!mpiexec -n 2 ./mpi-demo

I am rank 1 of 2: sum=1
I am rank 0 of 2: sum=1


* We can run more MPI processes than cores (or hardware threads), but you might need to use the `--oversubscribe` option because **oversubscription is usually expensive**
    * See our previous discussion of simultaneous multithreading

In [12]:
! mpirun -n 6 --oversubscribe ./mpi-demo

I am rank 1 of 6: sum=15
I am rank 4 of 6: sum=15
I am rank 0 of 6: sum=15
I am rank 3 of 6: sum=15
I am rank 2 of 6: sum=15
I am rank 5 of 6: sum=15


* You can use OpenMP within ranks of MPI (but use `MPI_Init_thread()`)
- Have to decide whether all the threads can make MPI calls, or only the master thread
- Leave future discussion of MPI+OpenMP to later (it's an interesting discussion)

## Point-to-point messaging

- The most basic and fundamental operation in MPI
    - "Transport some data to this rank"
```c
int MPI_Send(const void *buf, int count, MPI_Datatype datatype, int dest, int tag, MPI_Comm comm);
int MPI_Recv(void *buf, int count, MPI_Datatype datatype, int source, int tag, MPI_Comm comm, MPI_Status *status);
```

- `buf`: where the data is sent or recieved (must be allocated before hand)
- `count`: Number of entities in the buffer
- `datatype`: The kind of data in the buffer (`MPI_INT`, or `MPI_DOUBLE`)
    - (total size of buffer is `count * sizeof(datatype)`)
    - Can create custom MPI data types for whatever data is being transfered
    - Really only needs to know the size fo the data being transfered
- `source`/`dest`: the rank on the other end of the send/recv pair
    - Each rank needs to know who it's communicating with 
- `tag`: used to help differentiate different sends and recieves (MPI matches send and recv with the same tag)
- `status`: Confirms who sent the message recieved, what tag was recieved, and an error message
    - Can also pass `MPI_STATUS_IGNORE` instead if you don't it (not sure why this would be used)

- Both sides need to be aware of communication
    - Every `MPI_Send()` has to have a corresponding `MPI_Recv()`

**Ping Pong Example:**
```c
  double *A;
  int N;
  if (rank == 0) {
    MPI_Send(A, N, MPI_DOUBLE, 1, tag1, MPI_COMM_WORLD);
    MPI_Recv(A, N, MPI_DOUBLE, 1, tag2, MPI_COMM_WORLD, &stat);
  } else if (rank == 1) {
    MPI_Recv(A, N, MPI_DOUBLE, 0, tag1, MPI_COMM_WORLD, &stat);
    MPI_Send(A, N, MPI_DOUBLE, 0, tag2, MPI_COMM_WORLD);
```

In addition to collectives, MPI supports messaging directly between individual ranks.

![mpi send recieve](./LLNL_MPI_send_recieve_diagram.gif)

- Data sent over the network may arrive before rank is ready to recieve
- Data is stored in a "system buffer" managed by MPI
- `MPI_Recv()` then copies data from the system buffer to the rank's buffer

## Collective operations

MPI has a rich set of collective operations scoped by communicator, including the following.

![MPI collectives](./LLNL_MPI_collective_comm.gif)

```c
int MPI_Allreduce(const void *sendbuf, void *recvbuf, int count,
        MPI_Datatype datatype, MPI_Op op, MPI_Comm comm);
int MPI_Reduce(const void *sendbuf, void *recvbuf, int count,
        MPI_Datatype datatype, MPI_Op op, int root, MPI_Comm comm);
int MPI_Scan(const void *sendbuf, void *recvbuf, int count,
        MPI_Datatype datatype, MPI_Op op, MPI_Comm comm);
int MPI_Gather(const void *sendbuf, int sendcount, MPI_Datatype sendtype,
        void *recvbuf, int recvcount, MPI_Datatype recvtype, int root, MPI_Comm comm);
int MPI_Scatter(const void *sendbuf, int sendcount, MPI_Datatype sendtype,
        void *recvbuf, int recvcount, MPI_Datatype recvtype, int root, MPI_Comm comm);
```

* Implementations are optimized by vendors for their custom networks, and can be very fast.

![Fischer BGP](https://www.mcs.anl.gov/~fischer/gop/bgp_gop_png.png)

Notice how the time is basically independent of number of processes $P$, and only a small multiple of the cost to send a single message. Not all networks are this good.

- This is why you nearly always want to use the HPC-system-provided MPI library